In [ ]:
# Lab type: debug
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Retrieval Metrics: Recall@k, MRR, and nDCG
# Task: The evaluation harness below prints a flattering report. It
# contains 3 bugs — none of them crash. Find and fix each one, and write
# a one-sentence explanation after each fix.

# Lab: Debugging an Evaluation Harness

Evaluation bugs don't raise exceptions — they publish wrong numbers with confident names. The harness below was AI-generated to "measure our retriever". Every number it prints is wrong.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

## The harness under audit

In [ ]:
# --- AI-GENERATED EVALUATION HARNESS (contains 3 bugs) ---
# Review this code — is it correct?

def evaluate_recall(search_fn, eval_set, k=5):
    hits = 0
    for query, relevant in eval_set:
        results = [d for d, _ in search_fn(query, k=k)]
        if results[0] in relevant:            # <- look closely
            hits += 1
    return hits / len(eval_set)

def evaluate_mrr(search_fn, eval_set, k=5):
    reciprocal_ranks = []
    for query, relevant in eval_set:
        results = [d for d, _ in search_fn(query, k=k)]
        for i, doc in enumerate(results, start=1):
            if doc in relevant:
                reciprocal_ranks.append(1.0 / i)
                break
    # 'avoid division issues when a query has no hits'
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

# 'we didn't have labelled queries, so we generated them from the docs'
DOC_DERIVED_EVAL_SET = [(t[:60], {d}) for d, h, t in CORPUS[:8]]

# The team's real labelled set, which has drifted: one labelled source
# document ('integrations-guide') was deleted from the corpus last month.
TEAM_EVAL_SET = EVAL_SET + [
    ("does nimbus integrate with salesforce", {"integrations-guide"}),
]

print(f"recall@5: {evaluate_recall(dense_search, DOC_DERIVED_EVAL_SET):.2f}")
print(f"MRR:      {evaluate_mrr(dense_search, TEAM_EVAL_SET):.2f}")

## Bug 1: what does `evaluate_recall` actually compute?

Trace it by hand for one query. What metric is it, and what should recall@5 check?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** only `results[0]` is ever checked, so the function computes hit-rate@1 and labels it recall@5.

**Why it causes wrong behaviour:** every number in the report describes a stricter, different metric — comparisons against any recall@5 target or another system's recall are meaningless.

**Correct approach:**
```python
def recall_at_k(search_fn, eval_set, k=5):
    total = 0.0
    for query, relevant in eval_set:
        results = [d for d, _ in search_fn(query, k=k)]
        total += len(set(results[:k]) & relevant) / len(relevant)
    return total / len(eval_set)
```

</details>

## Bug 2: the MRR average

`TEAM_EVAL_SET` contains a query whose labelled source document no longer exists in the corpus — a realistic labelled-set drift. Trace what `evaluate_mrr` does with that query. Which population disappears from the average, and in which direction does the reported MRR move?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** a query with no relevant document in the top k appends nothing to `reciprocal_ranks`, so the mean is taken over successful queries only — the salesforce query silently vanishes and the reported MRR is unchanged by a total retrieval failure.

**Why it causes wrong behaviour:** the dropped zeros are exactly the retrieval failures — the population you most need to see. The reported MRR answers 'how well do we rank, when we succeed?' and inflates as the system fails more. A query with no relevant result scores 0 by definition; there is no division issue to avoid — count it as `0.0` (equivalently: divide the sum by `len(eval_set)`).

</details>

## Bug 3: where did the eval queries come from?

Look at `DOC_DERIVED_EVAL_SET`. What is being 'asked', and what will *any* retriever score on it? Compare with the phrasings in `EVAL_SET`.

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug:** each eval 'query' is the first 60 characters of the document's own text — a verbatim fragment of exactly what was embedded. This is the retrieval-evaluation analogue of evaluating on the training set: any retriever scores near-perfectly, and the number says nothing about real traffic.

**Why it causes wrong behaviour:** real users ask in their own words ("how do I get my money back"), which is precisely the vocabulary-mismatch problem retrieval must solve. Label real (or realistic) user phrasings, as `EVAL_SET` does.

</details>

## The fixed harness

Apply all three fixes and re-measure on the honest query set.

In [ ]:
# Fix for Bugs 1-3: correct recall, zeros kept in MRR, real query phrasings
def recall_at_k(search_fn, eval_set, k=5):
    total = 0.0
    for query, relevant in eval_set:
        results = [d for d, _ in search_fn(query, k=k)]
        total += len(set(results[:k]) & relevant) / len(relevant)
    return total / len(eval_set)

def mrr(search_fn, eval_set, k=5):
    total = 0.0
    for query, relevant in eval_set:
        results = [d for d, _ in search_fn(query, k=k)]
        rr = 0.0
        for i, doc in enumerate(results, start=1):
            if doc in relevant:
                rr = 1.0 / i
                break
        total += rr
    return total / len(eval_set)

print("Bug 2 made visible — same retriever, same queries:")
print(f"  buggy MRR (drops the failed query): "
      f"{evaluate_mrr(dense_search, TEAM_EVAL_SET):.2f}")
print(f"  fixed MRR (counts it as zero):      "
      f"{mrr(dense_search, TEAM_EVAL_SET):.2f}")
print()
print("Bug 3 made visible — doc-derived vs real phrasings:")
print(f"  doc-derived recall@5: {recall_at_k(dense_search, DOC_DERIVED_EVAL_SET):.2f}"
      "   <- flattering by construction")
print(f"  real-query recall@5:  {recall_at_k(dense_search, EVAL_SET):.2f}")
print(f"  real-query recall@1:  {recall_at_k(dense_search, EVAL_SET, k=1):.2f}"
      "   <- with recall@5, localises ranking-vs-retrieval faults")

## Summary

1. The buggy 'recall@5' was actually _______.
2. Dropping zero-scoring queries from MRR makes the metric measure only the queries where retrieval _______.
3. Eval queries derived from document titles are the retrieval analogue of evaluating on the _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **hit-rate@1**
2. **succeeded**
3. **training set**

</details>